# APS360 — Data Cleaning

This notebook walks through the data cleaning pipeline for `resume_data.csv`  
and produces `cleaned_resume_data.csv` that is used by both the baseline and primary models.

**Task**: Regression — predict `matched_score` (continuous float, 0–1)  
**Text embeddings**: SBERT (long text) + Word2Vec (skill lists) — applied at model time  
**Missing text**: zero-vector + `has_career_objective` binary flag

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_cleaning import clean_resume_data

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_columns', 40)
%matplotlib inline

## 1. Load raw data and inspect

In [ ]:
raw = pd.read_csv('../data/resume_data.csv')
print('Shape:', raw.shape)
raw.head(2)

In [ ]:
# Null counts per column
null_pct = (raw.isnull().sum() / len(raw) * 100).sort_values(ascending=False)
plt.figure(figsize=(12, 5))
null_pct.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Missingness per column (% of rows)')
plt.ylabel('% missing')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
print(null_pct.round(1).to_string())

In [ ]:
# Target variable distribution
plt.figure(figsize=(8, 4))
plt.hist(raw['matched_score'].dropna(), bins=40, color='teal', edgecolor='white')
plt.title('matched_score distribution (raw)')
plt.xlabel('matched_score')
plt.ylabel('count')
plt.tight_layout()
plt.show()
raw['matched_score'].describe()

## 2. Run the cleaning pipeline

In [ ]:
df = clean_resume_data(
    input_path='../data/resume_data.csv',
    output_path='../data/cleaned_resume_data.csv'
)

## 3. Inspect cleaned data

In [ ]:
print('Cleaned shape:', df.shape)
df.head(3)

In [ ]:
# Numeric feature summaries
num_cols = ['experience_min_years', 'age_min', 'age_max', 'years_since_graduation',
            'gpa_normalized', 'total_work_experience_years',
            'has_certification', 'num_certifications', 'has_career_objective',
            'matched_score']
df[num_cols].describe().round(3)

In [ ]:
# Distribution plots for numerical features
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(['experience_min_years', 'age_min', 'years_since_graduation',
                          'gpa_normalized', 'total_work_experience_years',
                          'has_certification', 'has_career_objective', 'matched_score']):
    axes[i].hist(df[col], bins=30, color='steelblue', edgecolor='white')
    axes[i].set_title(col)
    axes[i].set_xlabel('value')
plt.suptitle('Numerical feature distributions', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Categorical distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ['degree_level', 'result_type', 'has_career_objective']):
    vc = df[col].value_counts()
    ax.bar(vc.index, vc.values, color='teal', edgecolor='white')
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Top job positions
top_jobs = df['job_position_name'].value_counts().head(15)
plt.figure(figsize=(10, 4))
top_jobs.plot(kind='barh', color='steelblue')
plt.title('Top 15 job positions')
plt.xlabel('count')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for numeric features
plt.figure(figsize=(10, 8))
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Correlation matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Sample cleaned training example
sample = df.iloc[0]
for col in df.columns:
    val = sample[col]
    if isinstance(val, str) and len(val) > 100:
        val = val[:100] + '...'
    print(f'{col:35s}: {val}')

## 4. Column Fate Summary

| Category | Columns | Fate |
|---|---|---|
| **Dropped** | address, company_urls, online_links, responsibilities.1, professional_company_names, educational_institution_name, certification_providers, issue_dates, expiry_dates, languages, proficiency_levels, extra_curricular_* | Too many nulls / URL / duplicate |
| **Numerical** | experiencere_requirement → experience_min_years, age_requirement → age_min/age_max, passing_years → years_since_graduation, educational_results → gpa_normalized, start/end_dates → total_work_experience_years | Parsed from messy strings |
| **Binary flags** | career_objective → has_career_objective, certification_skills → has_certification | Missingness signal |
| **Text** | career_objective, skills, responsibilities, skills_required, educationaL_requirements, related_skils_in_job, major_field_of_studies | Cleaned; embedded at model time |
| **Categorical** | job_position_name, degree_level, result_type, most_recent_position | Normalized; one-hot at model time |
| **Target** | matched_score | Unchanged (continuous 0–1) |